# 🚀 Pertemuan 14 — Deploy ML App dengan Streamlit
### Mata Kuliah: Python Machine Learning

---

## Tujuan Pembelajaran

Setelah pertemuan ini, kamu bisa:
1. Menggunakan `@st.cache_data` dan `@st.cache_resource` untuk mempercepat aplikasi Streamlit
2. Mengelola state antar re-run menggunakan `st.session_state`
3. Membangun **multi-page app** dengan struktur `Home.py` + folder `pages/`
4. Membuat **full ML application** end-to-end: load data → train model → interaktif di web
5. Men-deploy aplikasi Streamlit ke **Streamlit Cloud** secara gratis

---

## Recap Cepat — Quiz Kilat ⚡

Sebelum mulai, jawab pertanyaan berikut (dari materi P13 Streamlit Dasar):

1. Apa fungsi `st.sidebar`? Bagaimana cara menambahkan widget di dalamnya?
2. Sebutkan **3 widget input** Streamlit yang kamu ingat dan kapan menggunakannya!
3. Apa yang terjadi jika user mengubah nilai slider di Streamlit — apa yang terjadi pada script?
4. Bagaimana cara membuat **dua kolom** berdampingan di Streamlit?
5. Apa perbedaan `st.write()` vs `st.dataframe()` vs `st.table()`?

---

In [1]:
import os

# Buat direktori untuk multi-page app
os.makedirs('../streamlit_apps/p14_ml_app/pages', exist_ok=True)
os.makedirs('../streamlit_apps/p14_ml_app/.streamlit', exist_ok=True)

print("✅ Direktori siap!")
print("Struktur yang dibuat:")
print("  streamlit_apps/")
print("  └── p14_ml_app/")
print("      ├── .streamlit/")
print("      │   └── config.toml")
print("      ├── pages/")
print("      │   ├── 1_EDA.py")
print("      │   ├── 2_Prediksi.py")
print("      │   └── 3_Tentang_Model.py")
print("      ├── Home.py")
print("      └── requirements.txt")

✅ Direktori siap!
Struktur yang dibuat:
  streamlit_apps/
  └── p14_ml_app/
      ├── .streamlit/
      │   └── config.toml
      ├── pages/
      │   ├── 1_EDA.py
      │   ├── 2_Prediksi.py
      │   └── 3_Tentang_Model.py
      ├── Home.py
      └── requirements.txt


---

## BAGIAN 1: Caching — `@st.cache_data` & `@st.cache_resource`

### Mengapa Caching Penting?

Setiap kali user mengubah widget (slider, dropdown, tombol), Streamlit **menjalankan ulang seluruh script** dari atas ke bawah. Ini masalah besar untuk operasi berat:

```
TANPA CACHE — lambat:                 DENGAN CACHE — cepat:
─────────────────────────────         ──────────────────────────────
User geser slider                     User geser slider
    ↓                                     ↓
Script dijalankan ulang               Script dijalankan ulang
    ↓                                     ↓
Load CSV (2 detik)         ←───────── Cek cache: sudah ada? ✅
Train model (10 detik)     ←───────── Ambil dari memory (0.001 detik)
Tampilkan hasil                       Tampilkan hasil
    ↓
Total: 12+ detik setiap interaksi     Total: ~0.01 detik
```

### `@st.cache_data` — Untuk Data & DataFrame

Digunakan untuk fungsi yang mengembalikan **data yang bisa di-serialize** (DataFrame, list, dict, numpy array, string, angka).

```python
import streamlit as st
import pandas as pd
from sklearn.datasets import fetch_california_housing

@st.cache_data
def load_data():
    """Fungsi ini hanya dijalankan SEKALI — hasilnya di-cache."""
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['MedHouseVal'] = housing.target
    return df

# Panggil berkali-kali → tetap cepat karena cache
df = load_data()   # pertama: 2 detik
df = load_data()   # kedua: 0.001 detik (dari cache)
```

**Kapan pakai `@st.cache_data`:**
| Use Case | Contoh |
|----------|--------|
| Load file CSV / Excel | `pd.read_csv(...)` |
| Fetch data dari API | `requests.get(url).json()` |
| Preprocessing data | `StandardScaler().fit_transform(X)` |
| Query database | `conn.execute("SELECT ...")` |

### `@st.cache_resource` — Untuk Model & Koneksi

Digunakan untuk objek yang **tidak bisa di-serialize** — objek berat yang dibagikan di semua user session.

```python
import streamlit as st
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

@st.cache_resource
def train_model():
    """Model hanya dilatih sekali, lalu di-share ke semua user."""
    housing = fetch_california_housing()
    X_train, X_test, y_train, y_test = train_test_split(
        housing.data, housing.target, test_size=0.2, random_state=42
    )
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model, X_test, y_test

model, X_test, y_test = train_model()   # hanya latih SEKALI
```

**Kapan pakai `@st.cache_resource`:**
| Use Case | Contoh |
|----------|--------|
| Model ML terlatih | `RandomForestRegressor`, `XGBClassifier` |
| Koneksi database | `psycopg2.connect(...)`, `SQLAlchemy engine` |
| Koneksi API client | `openai.Client(...)` |
| Large language model | `transformers.pipeline(...)` |

### Perbandingan Ringkas

| Aspek | `@st.cache_data` | `@st.cache_resource` |
|-------|-----------------|---------------------|
| **Untuk** | Data / DataFrame | Model / koneksi |
| **Copy data** | Ya (setiap panggilan dapat salinan) | Tidak (shared object) |
| **Thread-safe** | Ya | Tidak (hati-hati mutation) |
| **Serializable** | Harus serializable | Tidak harus |
| **Contoh return** | `pd.DataFrame`, `list`, `dict` | `sklearn model`, `DB connection` |

---

## BAGIAN 2: `st.session_state` — Variabel yang Persist Antar Re-run

### Masalah Tanpa `session_state`

Karena Streamlit menjalankan ulang seluruh script setiap ada interaksi, variabel biasa **tidak bisa menyimpan state**:

```python
# SALAH — counter selalu reset ke 0 setiap re-run!
count = 0
if st.button("Tambah"):
    count += 1
st.write(f"Count: {count}")  # selalu menampilkan 0 atau 1, tidak pernah lebih
```

### Solusi: `st.session_state`

`st.session_state` adalah **dictionary** yang persist selama session user aktif:

```python
# BENAR — counter terus bertambah
if 'count' not in st.session_state:
    st.session_state['count'] = 0   # inisialisasi sekali

if st.button("Tambah"):
    st.session_state['count'] += 1

st.write(f"Count: {st.session_state['count']}")   # terus bertambah!
```

### Contoh: History Prediksi

```python
import streamlit as st
import pandas as pd

# Inisialisasi history sebagai list of dict
if 'prediction_history' not in st.session_state:
    st.session_state['prediction_history'] = []

# Form input
income = st.slider("Median Income", 0.5, 15.0, 5.0)

if st.button("Prediksi"):
    result = income * 0.5 + 1.2   # contoh prediksi sederhana
    
    # Simpan ke history
    st.session_state['prediction_history'].append({
        'MedInc': income,
        'Prediksi (×$100K)': round(result, 3)
    })
    st.success(f"Harga: ${result * 100_000:,.0f}")

# Tampilkan history
if st.session_state['prediction_history']:
    st.subheader("Riwayat Prediksi")
    df_history = pd.DataFrame(st.session_state['prediction_history'])
    st.dataframe(df_history)
    
    if st.button("Hapus History"):
        st.session_state['prediction_history'] = []
        st.rerun()
```

### Kapan Pakai `session_state`

| Use Case | Contoh |
|----------|--------|
| Counter/toggle | Tombol klik yang terus bertambah |
| History prediksi | Menyimpan semua prediksi yang pernah dilakukan |
| Multi-step form | Wizard langkah 1 → 2 → 3 |
| Login state | Menyimpan info user yang sedang login |
| Keranjang belanja | Daftar item yang dipilih user |

---

## BAGIAN 3: Multi-page Apps

### Struktur Folder

Streamlit mendukung multi-page apps secara native. Cukup buat folder `pages/` di sebelah file utama:

```
p14_ml_app/
├── Home.py                  ← halaman utama (entry point)
├── pages/
│   ├── 1_EDA.py             ← halaman 1: Exploratory Data Analysis
│   ├── 2_Prediksi.py        ← halaman 2: Prediksi harga
│   └── 3_Tentang_Model.py   ← halaman 3: Info model
├── .streamlit/
│   └── config.toml          ← konfigurasi tema
└── requirements.txt         ← dependensi
```

Streamlit otomatis membuat **sidebar navigasi** dari file-file di folder `pages/`.

### Konvensi Penamaan File

```
Format: [nomor]_[Nama_Halaman].py

1_EDA.py              → tampil sebagai "EDA"
2_Prediksi.py         → tampil sebagai "Prediksi"
3_Tentang_Model.py    → tampil sebagai "Tentang Model"

Catatan:
  - Underscore (_) → spasi di sidebar
  - Nomor di depan → urutan di sidebar
  - Emoji bisa ditambahkan: 1_📊_EDA.py
```

### Navigasi Antar Halaman

```python
# Di Home.py — link ke halaman lain
st.page_link("pages/1_EDA.py", label="📊 Lihat EDA", icon="📊")
st.page_link("pages/2_Prediksi.py", label="🎯 Coba Prediksi", icon="🎯")

# Atau gunakan st.navigation (Streamlit >= 1.36)
pg = st.navigation([
    st.Page("Home.py", title="Beranda", icon="🏠"),
    st.Page("pages/1_EDA.py", title="EDA", icon="📊"),
    st.Page("pages/2_Prediksi.py", title="Prediksi", icon="🎯"),
])
pg.run()
```

### Tips Multi-page App

| Tips | Penjelasan |
|------|-----------|
| Shared cache | `@st.cache_data` dan `@st.cache_resource` di-share antar halaman |
| Session state | `st.session_state` persist saat pindah halaman |
| Config global | `.streamlit/config.toml` berlaku untuk semua halaman |
| Import bersama | Buat `utils.py` untuk fungsi yang dipakai banyak halaman |

---

## BAGIAN 4: Full ML App — Prediksi Harga Properti California

### Deskripsi Aplikasi

Kita akan membangun aplikasi ML lengkap untuk **memprediksi harga median rumah** di California berdasarkan data sensus.

```
Alur Aplikasi:
─────────────────────────────────────────────────────────
  [Home.py]                [pages/1_EDA.py]
  Welcome page        →    Eksplorasi data
  Dataset overview         Filter by income range
  Navigasi ke halaman      Distribusi, korelasi, peta
        ↓
  [pages/2_Prediksi.py]   [pages/3_Tentang_Model.py]
  Input fitur rumah   →    Performa model (MAE, RMSE, R²)
  Tombol prediksi          Feature importance
  Hasil harga prediksi     Info training
  History prediksi
```

### Dataset: California Housing

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
```

**Fitur Dataset:**

| Fitur | Deskripsi | Satuan |
|-------|-----------|--------|
| `MedInc` | Median pendapatan rumah tangga | ×$10,000 |
| `HouseAge` | Median usia rumah di blok | Tahun |
| `AveRooms` | Rata-rata jumlah kamar per rumah tangga | Kamar |
| `AveBedrms` | Rata-rata jumlah kamar tidur per rumah tangga | Kamar |
| `Population` | Jumlah penduduk di blok | Orang |
| `AveOccup` | Rata-rata jumlah penghuni per rumah tangga | Orang |
| `Latitude` | Garis lintang blok | Derajat |
| `Longitude` | Garis bujur blok | Derajat |
| `MedHouseVal` | **TARGET**: Median nilai rumah | ×$100,000 |

**Dataset**: 20,640 blok sensus dari California (1990)  
**Model**: Random Forest Regressor  
**Target range**: $14,999 – $500,001

In [2]:
%%writefile ../streamlit_apps/p14_ml_app/.streamlit/config.toml
[theme]
primaryColor = "#3498db"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f0f2f6"
textColor = "#262730"
font = "sans serif"

Writing ../streamlit_apps/p14_ml_app/.streamlit/config.toml


In [3]:
%%writefile ../streamlit_apps/p14_ml_app/requirements.txt
streamlit>=1.32.0
scikit-learn>=1.3.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.12.0

Writing ../streamlit_apps/p14_ml_app/requirements.txt


In [4]:
%%writefile ../streamlit_apps/p14_ml_app/Home.py
"""
Home.py — Halaman Utama Aplikasi Prediksi Harga Properti California
Pertemuan 14: Deploy ML App dengan Streamlit
"""

import streamlit as st
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

# ── Konfigurasi halaman ────────────────────────────────────────────────────────
st.set_page_config(
    page_title="California Housing Predictor",
    page_icon="🏠",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Load data untuk overview ──────────────────────────────────────────────────
@st.cache_data
def load_overview():
    """Load California Housing untuk statistik overview di halaman utama."""
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['MedHouseVal'] = housing.target
    return df

df = load_overview()

# ── Header ─────────────────────────────────────────────────────────────────────
st.title("🏠 California Housing Price Predictor")
st.markdown(
    """
    Selamat datang di aplikasi **Prediksi Harga Properti California** — 
    proyek Machine Learning end-to-end berbasis data sensus 1990.
    
    Aplikasi ini dibangun menggunakan **Streamlit** + **scikit-learn** sebagai
    bagian dari Pertemuan 14 mata kuliah Python Machine Learning.
    """
)

st.divider()

# ── Metrics overview ──────────────────────────────────────────────────────────
st.subheader("📊 Ringkasan Dataset")
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        label="Total Sampel",
        value=f"{len(df):,}",
        help="Jumlah blok sensus California"
    )
with col2:
    st.metric(
        label="Rata-rata Harga",
        value=f"${df['MedHouseVal'].mean() * 100_000:,.0f}",
        help="Median harga rumah dalam USD"
    )
with col3:
    st.metric(
        label="Harga Tertinggi",
        value=f"${df['MedHouseVal'].max() * 100_000:,.0f}",
        help="Harga median rumah tertinggi dalam dataset"
    )
with col4:
    st.metric(
        label="Jumlah Fitur",
        value="8",
        help="Fitur input untuk model prediksi"
    )

st.divider()

# ── Deskripsi dataset ─────────────────────────────────────────────────────────
st.subheader("📋 Tentang Dataset")
col_left, col_right = st.columns([1, 1])

with col_left:
    st.markdown(
        """
        **California Housing Dataset** berisi data dari sensus California 1990.
        Setiap baris merepresentasikan satu **blok sensus** — area kecil
        berpenduduk 600–3.000 orang.
        
        Dataset ini sering digunakan untuk belajar regresi karena:
        - Ukurannya cukup besar (20.640 sampel)
        - Fiturnya bermakna secara nyata (pendapatan, usia rumah, lokasi)
        - Target yang jelas (harga rumah dalam $100,000)
        """
    )

with col_right:
    fitur_desc = pd.DataFrame({
        'Fitur': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                  'Population', 'AveOccup', 'Latitude', 'Longitude'],
        'Deskripsi': [
            'Median pendapatan (×$10K)',
            'Median usia rumah (tahun)',
            'Rata-rata kamar per rumah tangga',
            'Rata-rata kamar tidur per rumah tangga',
            'Jumlah penduduk blok',
            'Rata-rata penghuni per rumah tangga',
            'Garis lintang blok',
            'Garis bujur blok',
        ]
    })
    st.dataframe(fitur_desc, use_container_width=True, hide_index=True)

st.divider()

# ── Navigasi ke halaman lain ──────────────────────────────────────────────────
st.subheader("🗺️ Navigasi Aplikasi")
nav_col1, nav_col2, nav_col3 = st.columns(3)

with nav_col1:
    st.markdown(
        """
        ### 📊 EDA
        Eksplorasi distribusi data, korelasi antar fitur,
        dan peta geografis harga rumah di California.
        """
    )
    st.page_link("pages/1_EDA.py", label="Buka halaman EDA →", icon="📊")

with nav_col2:
    st.markdown(
        """
        ### 🎯 Prediksi
        Masukkan nilai fitur rumah dan dapatkan prediksi
        harga dari model Random Forest secara real-time.
        """
    )
    st.page_link("pages/2_Prediksi.py", label="Buka halaman Prediksi →", icon="🎯")

with nav_col3:
    st.markdown(
        """
        ### 🤖 Tentang Model
        Lihat performa model (MAE, RMSE, R²),
        feature importance, dan detail training.
        """
    )
    st.page_link("pages/3_Tentang_Model.py", label="Buka halaman Model →", icon="🤖")

st.divider()

# ── Footer ─────────────────────────────────────────────────────────────────────
st.caption(
    "🎓 Dibuat untuk Pertemuan 14 — Python Machine Learning | "
    "Dataset: sklearn.datasets.fetch_california_housing | "
    "Model: RandomForestRegressor"
)

# Animasi sambutan
if 'welcomed' not in st.session_state:
    st.balloons()
    st.session_state['welcomed'] = True

Writing ../streamlit_apps/p14_ml_app/Home.py


In [5]:
%%writefile ../streamlit_apps/p14_ml_app/pages/1_EDA.py
"""
1_EDA.py — Halaman Exploratory Data Analysis
Pertemuan 14: Deploy ML App dengan Streamlit
"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# ── Konfigurasi halaman ────────────────────────────────────────────────────────
st.set_page_config(
    page_title="EDA — California Housing",
    page_icon="📊",
    layout="wide",
)

# ── Load data dengan cache ────────────────────────────────────────────────────
@st.cache_data
def load_data():
    """Load California Housing dataset — di-cache agar tidak reload setiap interaksi."""
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['MedHouseVal'] = housing.target
    return df

df = load_data()

# ── Header ─────────────────────────────────────────────────────────────────────
st.title("📊 Exploratory Data Analysis")
st.markdown("Eksplorasi dataset **California Housing** — distribusi, korelasi, dan peta geografis harga rumah.")

# ── Sidebar: Filter ───────────────────────────────────────────────────────────
st.sidebar.header("🔧 Filter Data")
st.sidebar.markdown("Filter dataset berdasarkan rentang median pendapatan:")

inc_min = float(df['MedInc'].min())
inc_max = float(df['MedInc'].max())

income_range = st.sidebar.slider(
    "Median Income (×$10K)",
    min_value=inc_min,
    max_value=inc_max,
    value=(inc_min, inc_max),
    step=0.1,
    help="Filter blok sensus berdasarkan rentang median pendapatan"
)

# Terapkan filter
df_filtered = df[
    (df['MedInc'] >= income_range[0]) &
    (df['MedInc'] <= income_range[1])
].copy()

st.sidebar.markdown(f"**Data setelah filter:** {len(df_filtered):,} dari {len(df):,} blok")

# ── Metrics ringkas ───────────────────────────────────────────────────────────
st.subheader("Statistik Ringkas (Data Terfilter)")
m1, m2, m3, m4 = st.columns(4)

with m1:
    st.metric("Rata-rata Harga", f"${df_filtered['MedHouseVal'].mean() * 100_000:,.0f}")
with m2:
    st.metric("Median Harga", f"${df_filtered['MedHouseVal'].median() * 100_000:,.0f}")
with m3:
    st.metric("Harga Tertinggi", f"${df_filtered['MedHouseVal'].max() * 100_000:,.0f}")
with m4:
    st.metric("Harga Terendah", f"${df_filtered['MedHouseVal'].min() * 100_000:,.0f}")

st.divider()

# ── Tabs ──────────────────────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["📈 Distribusi Target", "🔗 Korelasi Fitur", "🗺️ Peta Geografis"])

# ── Tab 1: Distribusi target ──────────────────────────────────────────────────
with tab1:
    st.subheader("Distribusi Harga Rumah (MedHouseVal)")

    col_hist, col_box = st.columns([2, 1])

    with col_hist:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(
            df_filtered['MedHouseVal'] * 100_000,
            bins=50,
            color='#3498db',
            edgecolor='white',
            alpha=0.85
        )
        ax.axvline(
            df_filtered['MedHouseVal'].mean() * 100_000,
            color='#e74c3c', linestyle='--', linewidth=2,
            label=f"Rata-rata: ${df_filtered['MedHouseVal'].mean() * 100_000:,.0f}"
        )
        ax.axvline(
            df_filtered['MedHouseVal'].median() * 100_000,
            color='#f39c12', linestyle='--', linewidth=2,
            label=f"Median: ${df_filtered['MedHouseVal'].median() * 100_000:,.0f}"
        )
        ax.set_xlabel("Harga Rumah (USD)", fontsize=11)
        ax.set_ylabel("Jumlah Blok Sensus", fontsize=11)
        ax.set_title("Distribusi Harga Median Rumah", fontsize=13)
        ax.legend()
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        plt.tight_layout()
        st.pyplot(fig)
        plt.close()

    with col_box:
        st.markdown("**Statistik Deskriptif:**")
        desc = df_filtered['MedHouseVal'].describe() * 100_000
        desc.index = ['Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']
        desc_df = desc.reset_index()
        desc_df.columns = ['Statistik', 'Harga (USD)']
        desc_df['Harga (USD)'] = desc_df['Harga (USD)'].apply(lambda x: f"${x:,.0f}")
        st.dataframe(desc_df, use_container_width=True, hide_index=True)

        st.markdown(
            """
            > **Catatan:** Banyak nilai di atas $500,000 kemungkinan 
            > merupakan nilai yang di-cap (di-ceiling) oleh dataset.
            """
        )

# ── Tab 2: Correlation heatmap ────────────────────────────────────────────────
with tab2:
    st.subheader("Korelasi Antar Fitur")
    st.markdown(
        "Heatmap korelasi Pearson menunjukkan seberapa kuat hubungan linear "
        "antar fitur. Nilai mendekati **+1** atau **-1** = korelasi kuat."
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    corr_matrix = df_filtered.corr()

    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap='RdYlBu_r',
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        ax=ax,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title("Correlation Heatmap — California Housing", fontsize=13)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

    # Temuan utama
    st.subheader("Temuan Utama")
    col_f1, col_f2 = st.columns(2)
    with col_f1:
        st.success(
            "**MedInc** memiliki korelasi positif tertinggi dengan harga rumah (+0.69) — "
            "semakin tinggi pendapatan median, semakin mahal harga rumah."
        )
    with col_f2:
        st.info(
            "**Latitude** dan **Longitude** berkorelasi negatif kuat dengan harga — "
            "wilayah selatan (Latitude rendah) dan pesisir (Longitude tinggi) lebih mahal."
        )

# ── Tab 3: Geographic scatter ─────────────────────────────────────────────────
with tab3:
    st.subheader("Peta Geografis Harga Rumah California")
    st.markdown(
        "Setiap titik adalah satu blok sensus. "
        "**Warna** menunjukkan harga: 🔵 murah → 🔴 mahal."
    )

    # Sample untuk performa (max 5000 titik)
    sample_size = min(5000, len(df_filtered))
    df_sample = df_filtered.sample(n=sample_size, random_state=42)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        df_sample['Longitude'],
        df_sample['Latitude'],
        c=df_sample['MedHouseVal'],
        cmap='RdYlBu_r',
        alpha=0.5,
        s=10,
        vmin=df_filtered['MedHouseVal'].quantile(0.05),
        vmax=df_filtered['MedHouseVal'].quantile(0.95),
    )
    plt.colorbar(scatter, ax=ax, label='Harga Rumah (×$100K)', shrink=0.8)

    # Anotasi kota besar
    kota = {
        'San Francisco': (-122.4, 37.8),
        'Los Angeles':   (-118.2, 34.1),
        'San Diego':     (-117.1, 32.7),
        'Sacramento':    (-121.5, 38.6),
    }
    for nama, (lon, lat) in kota.items():
        ax.annotate(
            nama,
            xy=(lon, lat),
            fontsize=9,
            color='black',
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
        )

    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)
    ax.set_title(
        f"Peta Harga Rumah California ({sample_size:,} blok sampel)\n"
        "Merah = Mahal | Biru = Murah",
        fontsize=13
    )
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

    st.caption(
        f"Menampilkan {sample_size:,} dari {len(df_filtered):,} blok sensus "
        "(disample untuk performa)."
    )

Writing ../streamlit_apps/p14_ml_app/pages/1_EDA.py


In [6]:
%%writefile ../streamlit_apps/p14_ml_app/pages/2_Prediksi.py
"""
2_Prediksi.py — Halaman Prediksi Harga Rumah
Pertemuan 14: Deploy ML App dengan Streamlit
"""

import streamlit as st
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# ── Konfigurasi halaman ────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Prediksi — California Housing",
    page_icon="🎯",
    layout="wide",
)

# ── Load data & train model (di-cache) ────────────────────────────────────────
@st.cache_data
def load_data():
    """Load California Housing sebagai DataFrame — di-cache."""
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['MedHouseVal'] = housing.target
    return df, housing.feature_names

@st.cache_resource
def train_model():
    """
    Latih Random Forest Regressor pada California Housing.
    Di-cache dengan @st.cache_resource karena model sklearn tidak serializable.
    Hanya dijalankan SEKALI — semua user berbagi model yang sama.
    """
    housing = fetch_california_housing()
    X = housing.data
    y = housing.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Training model Random Forest
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1   # pakai semua CPU core
    )
    model.fit(X_train, y_train)

    return model, X_train, X_test, y_train, y_test

# Load data dan model
df, feature_names = load_data()
model, X_train, X_test, y_train, y_test = train_model()

# Rata-rata dataset untuk perbandingan
df_mean = df[list(feature_names)].mean()

# ── Header ─────────────────────────────────────────────────────────────────────
st.title("🎯 Prediksi Harga Rumah")
st.markdown(
    "Masukkan karakteristik blok sensus di sidebar, lalu klik **Prediksi** "
    "untuk mendapatkan estimasi harga median rumah dari model Random Forest."
)

# ── Sidebar: Input fitur ──────────────────────────────────────────────────────
st.sidebar.header("🏠 Input Fitur Rumah")
st.sidebar.markdown("Sesuaikan nilai fitur blok sensus:")

med_inc = st.sidebar.slider(
    "MedInc — Median Pendapatan (×$10K)",
    min_value=0.5, max_value=15.0,
    value=float(df_mean['MedInc']),
    step=0.1,
    help="Median pendapatan rumah tangga dalam blok sensus (dalam kelipatan $10,000)"
)

house_age = st.sidebar.slider(
    "HouseAge — Median Usia Rumah (tahun)",
    min_value=1.0, max_value=52.0,
    value=float(df_mean['HouseAge']),
    step=1.0,
    help="Median usia rumah dalam blok sensus"
)

ave_rooms = st.sidebar.slider(
    "AveRooms — Rata-rata Jumlah Kamar",
    min_value=1.0, max_value=10.0,
    value=float(min(df_mean['AveRooms'], 10.0)),
    step=0.1,
    help="Rata-rata jumlah kamar per rumah tangga"
)

ave_bedrms = st.sidebar.slider(
    "AveBedrms — Rata-rata Kamar Tidur",
    min_value=1.0, max_value=5.0,
    value=float(min(df_mean['AveBedrms'], 5.0)),
    step=0.1,
    help="Rata-rata jumlah kamar tidur per rumah tangga"
)

population = st.sidebar.number_input(
    "Population — Jumlah Penduduk Blok",
    min_value=3, max_value=35682,
    value=int(df_mean['Population']),
    step=100,
    help="Jumlah total penduduk dalam blok sensus"
)

ave_occup = st.sidebar.slider(
    "AveOccup — Rata-rata Penghuni per Rumah",
    min_value=1.0, max_value=10.0,
    value=float(min(df_mean['AveOccup'], 10.0)),
    step=0.1,
    help="Rata-rata jumlah penghuni per rumah tangga"
)

latitude = st.sidebar.slider(
    "Latitude — Garis Lintang",
    min_value=32.5, max_value=42.0,
    value=float(df_mean['Latitude']),
    step=0.1,
    help="Garis lintang blok sensus (32 = selatan, 42 = utara California)"
)

longitude = st.sidebar.slider(
    "Longitude — Garis Bujur",
    min_value=-124.0, max_value=-114.0,
    value=float(df_mean['Longitude']),
    step=0.1,
    help="Garis bujur blok sensus (-124 = barat pesisir, -114 = timur)"
)

# Susun input sebagai array untuk prediksi
input_features = np.array([[
    med_inc, house_age, ave_rooms, ave_bedrms,
    population, ave_occup, latitude, longitude
]])

# ── Tombol prediksi ──────────────────────────────────────────────────────────
st.subheader("Hasil Prediksi")

col_pred, col_info = st.columns([1, 2])

with col_pred:
    if st.button("🔮 Prediksi Sekarang", type="primary", use_container_width=True):
        # Jalankan prediksi
        predicted_val = model.predict(input_features)[0]
        predicted_usd = predicted_val * 100_000

        # Simpan ke session_state
        if 'prediction_history' not in st.session_state:
            st.session_state['prediction_history'] = []

        st.session_state['prediction_history'].append({
            'MedInc': med_inc,
            'HouseAge': house_age,
            'AveRooms': ave_rooms,
            'AveBedrms': ave_bedrms,
            'Population': population,
            'AveOccup': ave_occup,
            'Latitude': latitude,
            'Longitude': longitude,
            'Prediksi (×$100K)': round(predicted_val, 3),
            'Prediksi (USD)': f"${predicted_usd:,.0f}"
        })

        st.session_state['last_prediction'] = predicted_val

    # Tampilkan hasil jika ada
    if 'last_prediction' in st.session_state:
        pred_val = st.session_state['last_prediction']
        pred_usd = pred_val * 100_000

        st.metric(
            label="Harga Prediksi",
            value=f"${pred_usd:,.0f}",
            help=f"Nilai mentah: {pred_val:.4f} × $100,000"
        )

        # Pesan berdasarkan nilai prediksi
        if pred_usd > 350_000:
            st.success("🏖️ Properti premium — di atas rata-rata California!")
        elif pred_usd > 200_000:
            st.info("🏡 Properti sedang — harga tipikal California.")
        else:
            st.warning("🏘️ Properti terjangkau — di bawah rata-rata California.")

with col_info:
    st.markdown("**Perbandingan Input vs Rata-rata Dataset:**")

    # Tabel perbandingan
    comparison_data = {
        'Fitur': list(feature_names),
        'Nilai Input': [
            med_inc, house_age, ave_rooms, ave_bedrms,
            population, ave_occup, latitude, longitude
        ],
        'Rata-rata Dataset': [round(df_mean[f], 3) for f in feature_names],
    }
    df_comp = pd.DataFrame(comparison_data)
    df_comp['Selisih'] = (df_comp['Nilai Input'] - df_comp['Rata-rata Dataset']).round(3)
    df_comp['Status'] = df_comp['Selisih'].apply(
        lambda x: '▲ Di atas rata-rata' if x > 0 else ('▼ Di bawah rata-rata' if x < 0 else '= Rata-rata')
    )
    st.dataframe(df_comp, use_container_width=True, hide_index=True)

st.divider()

# ── History prediksi ───────────────────────────────────────────────────────────
st.subheader("📋 Riwayat Prediksi")

if 'prediction_history' not in st.session_state or not st.session_state['prediction_history']:
    st.info("Belum ada prediksi. Klik tombol **Prediksi Sekarang** untuk memulai.")
else:
    df_history = pd.DataFrame(st.session_state['prediction_history'])
    st.dataframe(df_history, use_container_width=True, hide_index=True)
    st.caption(f"Total prediksi: {len(df_history)} kali")

    col_clear, col_dl = st.columns([1, 3])
    with col_clear:
        if st.button("🗑️ Hapus History", use_container_width=True):
            st.session_state['prediction_history'] = []
            if 'last_prediction' in st.session_state:
                del st.session_state['last_prediction']
            st.rerun()
    with col_dl:
        csv_data = df_history.to_csv(index=False).encode('utf-8')
        st.download_button(
            label="📥 Download History sebagai CSV",
            data=csv_data,
            file_name="prediction_history.csv",
            mime="text/csv",
            use_container_width=True
        )

Writing ../streamlit_apps/p14_ml_app/pages/2_Prediksi.py


In [7]:
%%writefile ../streamlit_apps/p14_ml_app/pages/3_Tentang_Model.py
"""
3_Tentang_Model.py — Halaman Performa & Info Model
Pertemuan 14: Deploy ML App dengan Streamlit
"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Konfigurasi halaman ────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Tentang Model — California Housing",
    page_icon="🤖",
    layout="wide",
)

# ── Load & train model (di-cache — shared dengan halaman Prediksi) ─────────────
@st.cache_resource
def train_model():
    """
    Latih model dan kembalikan model + split data untuk evaluasi.
    @st.cache_resource: model di-share antar halaman dan antar user.
    """
    housing = fetch_california_housing()
    X = housing.data
    y = housing.target
    feature_names = housing.feature_names

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    return model, X_train, X_test, y_train, y_test, feature_names

model, X_train, X_test, y_train, y_test, feature_names = train_model()

# ── Hitung metrik performa ─────────────────────────────────────────────────────
@st.cache_data
def compute_metrics(_model, _X_train, _X_test, _y_train, _y_test):
    """Hitung MAE, RMSE, R² untuk train dan test set."""
    y_pred_train = _model.predict(_X_train)
    y_pred_test  = _model.predict(_X_test)

    metrics = {
        'train': {
            'MAE':  mean_absolute_error(_y_train, y_pred_train),
            'RMSE': np.sqrt(mean_squared_error(_y_train, y_pred_train)),
            'R2':   r2_score(_y_train, y_pred_train),
        },
        'test': {
            'MAE':  mean_absolute_error(_y_test, y_pred_test),
            'RMSE': np.sqrt(mean_squared_error(_y_test, y_pred_test)),
            'R2':   r2_score(_y_test, y_pred_test),
        },
        'y_pred_test': y_pred_test,
    }
    return metrics

metrics = compute_metrics(model, X_train, X_test, y_train, y_test)

# ── Header ─────────────────────────────────────────────────────────────────────
st.title("🤖 Tentang Model")
st.markdown(
    "Halaman ini menampilkan performa **Random Forest Regressor** "
    "yang digunakan untuk memprediksi harga rumah California."
)

# ── Info training ──────────────────────────────────────────────────────────────
st.subheader("⚙️ Konfigurasi Training")

cfg_col1, cfg_col2, cfg_col3 = st.columns(3)
with cfg_col1:
    st.info(f"**Algoritma:** Random Forest Regressor")
    st.info(f"**n_estimators:** 100 pohon")
with cfg_col2:
    st.info(f"**max_depth:** 15")
    st.info(f"**min_samples_split:** 5")
with cfg_col3:
    st.info(f"**Training size:** {len(X_train):,} sampel (80%)")
    st.info(f"**Test size:** {len(X_test):,} sampel (20%)")

st.divider()

# ── Metrik performa ────────────────────────────────────────────────────────────
st.subheader("📈 Performa Model")

m1, m2, m3 = st.columns(3)

with m1:
    st.metric(
        label="MAE (Test)",
        value=f"{metrics['test']['MAE']:.4f} × $100K",
        delta=f"Train: {metrics['train']['MAE']:.4f}",
        delta_color="inverse",
        help="Mean Absolute Error — rata-rata selisih absolut prediksi vs aktual"
    )
    st.caption(f"= ~${metrics['test']['MAE'] * 100_000:,.0f} rata-rata error")

with m2:
    st.metric(
        label="RMSE (Test)",
        value=f"{metrics['test']['RMSE']:.4f} × $100K",
        delta=f"Train: {metrics['train']['RMSE']:.4f}",
        delta_color="inverse",
        help="Root Mean Squared Error — memberi penalti lebih besar untuk error besar"
    )
    st.caption(f"= ~${metrics['test']['RMSE'] * 100_000:,.0f} RMSE")

with m3:
    st.metric(
        label="R² Score (Test)",
        value=f"{metrics['test']['R2']:.4f}",
        delta=f"Train: {metrics['train']['R2']:.4f}",
        help="R² = proporsi variansi target yang dijelaskan model (1.0 = sempurna)"
    )
    st.caption(f"Model menjelaskan {metrics['test']['R2']*100:.1f}% variansi harga")

# Interpretasi metrik
st.markdown("**Interpretasi Metrik:**")
col_interp1, col_interp2 = st.columns(2)
with col_interp1:
    if metrics['test']['R2'] > 0.80:
        st.success(f"R² = {metrics['test']['R2']:.3f} — Model sangat baik (> 0.80)!")
    elif metrics['test']['R2'] > 0.60:
        st.info(f"R² = {metrics['test']['R2']:.3f} — Model cukup baik (0.60–0.80).")
    else:
        st.warning(f"R² = {metrics['test']['R2']:.3f} — Model perlu ditingkatkan (< 0.60).")

with col_interp2:
    gap_r2 = metrics['train']['R2'] - metrics['test']['R2']
    if gap_r2 > 0.10:
        st.warning(f"Gap Train/Test R² = {gap_r2:.3f} — Ada indikasi sedikit overfitting.")
    else:
        st.success(f"Gap Train/Test R² = {gap_r2:.3f} — Model generalisasi dengan baik!")

st.divider()

# ── Actual vs Predicted plot ────────────────────────────────────────────────────
st.subheader("🎯 Aktual vs Prediksi (Test Set)")

# Sample 1000 untuk visualisasi
idx_sample = np.random.RandomState(42).choice(len(y_test), min(1000, len(y_test)), replace=False)
y_actual_sample   = y_test[idx_sample]
y_pred_sample     = metrics['y_pred_test'][idx_sample]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    y_actual_sample, y_pred_sample,
    alpha=0.4, s=15, color='#3498db', label='Prediksi'
)
# Garis ideal (y = x)
lims = [min(y_actual_sample.min(), y_pred_sample.min()),
        max(y_actual_sample.max(), y_pred_sample.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Prediksi sempurna (y=x)')
ax.set_xlabel("Harga Aktual (×$100K)", fontsize=11)
ax.set_ylabel("Harga Prediksi (×$100K)", fontsize=11)
ax.set_title(
    f"Aktual vs Prediksi — Random Forest\nR² = {metrics['test']['R2']:.4f} | RMSE = {metrics['test']['RMSE']:.4f}",
    fontsize=13
)
ax.legend()
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
st.pyplot(fig)
plt.close()

st.caption(
    "Titik-titik yang mendekati garis merah putus-putus = prediksi akurat. "
    "Penyebaran jauh dari garis = error prediksi."
)

st.divider()

# ── Feature importance ─────────────────────────────────────────────────────────
st.subheader("📊 Feature Importance")
st.markdown(
    "Feature importance menunjukkan kontribusi relatif setiap fitur "
    "dalam pengambilan keputusan model Random Forest."
)

importances = model.feature_importances_
df_imp = pd.DataFrame({
    'Fitur': list(feature_names),
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db' if imp > importances.mean() else '#bdc3c7' for imp in df_imp['Importance']]
bars = ax.barh(df_imp['Fitur'], df_imp['Importance'], color=colors, edgecolor='white')
ax.axvline(importances.mean(), color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Rata-rata importance = {importances.mean():.3f}')
ax.set_xlabel("Feature Importance (Mean Decrease Impurity)", fontsize=11)
ax.set_title("Feature Importance — Random Forest Regressor", fontsize=13)
ax.legend()

for bar, val in zip(bars, df_imp['Importance']):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}',
        va='center', fontsize=9
    )

plt.tight_layout()
st.pyplot(fig)
plt.close()

st.divider()

# ── Tabel deskripsi fitur ─────────────────────────────────────────────────────
st.subheader("📋 Deskripsi Fitur Dataset")

fitur_desc = pd.DataFrame({
    'Fitur': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
              'Population', 'AveOccup', 'Latitude', 'Longitude'],
    'Deskripsi Lengkap': [
        'Median pendapatan rumah tangga dalam blok sensus (dalam kelipatan $10,000)',
        'Median usia rumah dalam blok sensus (tahun)',
        'Rata-rata jumlah kamar total per rumah tangga dalam blok',
        'Rata-rata jumlah kamar tidur per rumah tangga dalam blok',
        'Jumlah total penduduk dalam blok sensus',
        'Rata-rata jumlah penghuni per rumah tangga dalam blok',
        'Garis lintang blok (32° = selatan, 42° = utara California)',
        'Garis bujur blok (-124° = barat pesisir, -114° = timur pedalaman)',
    ],
    'Satuan': [
        '×$10,000', 'Tahun', 'Kamar', 'Kamar tidur',
        'Orang', 'Orang/rumah', 'Derajat', 'Derajat'
    ],
    'Importance': [f"{importances[i]:.3f}" for i in range(len(feature_names))]
})

st.dataframe(fitur_desc, use_container_width=True, hide_index=True)

Writing ../streamlit_apps/p14_ml_app/pages/3_Tentang_Model.py


---

## BAGIAN 5: Deploy ke Streamlit Cloud

### Pre-requisites

Sebelum deploy, pastikan kamu sudah punya:
- Akun **GitHub** (gratis di github.com)
- Akun **Streamlit Cloud** (gratis di streamlit.io/cloud — login dengan GitHub)
- File `requirements.txt` yang lengkap di root project

---

### Langkah 1: Siapkan Repository GitHub

Struktur repository yang dibutuhkan:

```
my-ml-app/                         ← root repository
├── Home.py                        ← file utama (entry point)
├── pages/
│   ├── 1_EDA.py
│   ├── 2_Prediksi.py
│   └── 3_Tentang_Model.py
├── .streamlit/
│   └── config.toml
└── requirements.txt               ← WAJIB ADA
```

**Command untuk upload ke GitHub:**
```bash
# Inisialisasi git di folder app
cd streamlit_apps/p14_ml_app
git init
git add .
git commit -m "Initial commit: California Housing ML App"

# Buat repo baru di GitHub, lalu:
git remote add origin https://github.com/username/california-housing-app.git
git branch -M main
git push -u origin main
```

> **Tips:** Pastikan `requirements.txt` sudah berisi semua library yang dipakai.
> Jika ada library yang kurang, app akan gagal saat startup di cloud.

---

### Langkah 2: Login ke Streamlit Cloud

```
[ Buka browser → streamlit.io/cloud ]
         ↓
[ Klik "Sign in with GitHub" ]
         ↓
[ Authorize Streamlit untuk akses repositorimu ]
         ↓
[ Kamu akan diarahkan ke dashboard Streamlit Cloud ]
```

*[Screenshot: Tampilan login Streamlit Cloud]*

---

### Langkah 3: Deploy App Baru

```
[ Dashboard Streamlit Cloud ]
         ↓
[ Klik tombol "New app" (pojok kanan atas) ]
         ↓
[ Isi form deployment: ]
  Repository  : username/california-housing-app
  Branch      : main
  Main file   : Home.py              ← FILE UTAMA, bukan pages/
         ↓
[ Klik "Deploy!" ]
         ↓
[ Tunggu build (2-5 menit pertama kali) ]
         ↓
[ App live di: username-california-housing-app.streamlit.app ]
```

*[Screenshot: Form deployment Streamlit Cloud]*

---

### Langkah 4: Share & Update

```
Setelah deploy berhasil:

  URL app kamu: https://username-appname.streamlit.app

  Update app:
    1. Edit file lokal
    2. git add . && git commit -m "Update"
    3. git push
    4. Streamlit Cloud otomatis rebuild! (1-2 menit)
```

---

### Checklist Troubleshooting Umum

| Error | Penyebab | Solusi |
|-------|----------|--------|
| `ModuleNotFoundError: No module named 'X'` | Library tidak ada di `requirements.txt` | Tambahkan `X>=versi` ke `requirements.txt` |
| `MemoryError` | Model terlalu besar (> ~800MB RAM Streamlit Cloud free) | Gunakan model lebih kecil, kurangi `n_estimators` |
| App sangat lambat startup | Model di-train setiap startup | Gunakan `@st.cache_resource` yang benar |
| `FileNotFoundError` | Path file relatif tidak ditemukan | Gunakan path relatif dari root repo |
| App crash setelah idle | Streamlit Cloud hibernate app gratis | Normal — akan restart saat ada pengunjung baru |
| `AttributeError` untuk sklearn | Versi sklearn berbeda | Pin versi: `scikit-learn==1.3.0` |

---

### Alternatif Platform Deploy

| Platform | Gratis? | Keunggulan | Cocok untuk |
|----------|---------|------------|-------------|
| **Streamlit Cloud** | Ya | Paling mudah untuk app Streamlit | App Streamlit apapun |
| **Hugging Face Spaces** | Ya | Support Streamlit + Gradio + Docker | ML/AI apps |
| **Railway** | Trial $5 | Fleksibel, support semua framework | App web umum |
| **Render** | Ya (terbatas) | Support web service, cron job | Backend + frontend |
| **Google Cloud Run** | Ya (terbatas) | Scalable, kontrol penuh | Production enterprise |

---

## ✏️ Latihan Mandiri

Kerjakan latihan berikut untuk memperkuat pemahaman tentang Streamlit deploy dan full ML app.

---

### Latihan 1: Tambahkan Halaman "About / Info Tim"

Buat halaman baru `pages/4_About.py` yang berisi informasi tim.
Gunakan `st.columns`, `st.image` (atau emoji sebagai placeholder), dan `st.markdown`.


In [ ]:
%%writefile ../streamlit_apps/p14_ml_app/pages/4_About.py
"""
LATIHAN 1: Halaman About / Info Tim
Lengkapi kode di bawah ini dengan informasi tim kamu!
"""

import streamlit as st

st.set_page_config(page_title="About — Tim ML", page_icon="👥", layout="wide")

st.title("👥 Tentang Tim")
st.markdown("Halaman ini dibuat sebagai bagian dari **Latihan 1 Pertemuan 14**.")

st.divider()

# ── TODO: Isi dengan info tim kamu ────────────────────────────────────────────
# Ganti nama, NIM, dan deskripsi sesuai tim kamu

anggota = [
    {"nama": "Anggota 1", "nim": "NIM-001", "peran": "ML Engineer", "kontribusi": "Model training & evaluasi"},
    {"nama": "Anggota 2", "nim": "NIM-002", "peran": "Data Analyst", "kontribusi": "EDA & visualisasi"},
    {"nama": "Anggota 3", "nim": "NIM-003", "peran": "Frontend Dev", "kontribusi": "UI/UX Streamlit"},
]

st.subheader("Anggota Tim")
cols = st.columns(len(anggota))

for col, orang in zip(cols, anggota):
    with col:
        # Ganti dengan st.image("path_foto.jpg") jika ada foto
        st.markdown(f"### 👤 {orang['nama']}")
        st.markdown(f"**NIM:** {orang['nim']}")
        st.markdown(f"**Peran:** {orang['peran']}")
        st.markdown(f"**Kontribusi:** {orang['kontribusi']}")

st.divider()

# ── TODO: Tambahkan deskripsi project ─────────────────────────────────────────
st.subheader("Tentang Project")
st.markdown(
    """
    **Nama Project:** California Housing Price Predictor
    
    **Deskripsi:** *(Tulis deskripsi singkat project tim kamu di sini)*
    
    **Dataset:** California Housing Dataset (sklearn)
    
    **Teknologi:** Python, Streamlit, scikit-learn, pandas, matplotlib
    
    **Link Repo GitHub:** *(Isi link GitHub repo tim kamu)*
    """
)

st.info("Setelah mengisi semua informasi, deploy app ini ke Streamlit Cloud dan submit URL-nya!")

---

### Latihan 2: Simpan History Prediksi ke CSV dengan `session_state`

Modifikasi halaman prediksi (`2_Prediksi.py`) agar:
1. Setiap prediksi otomatis disimpan ke `st.session_state`
2. Ada tombol **"Export ke CSV"** yang menyimpan history ke file `riwayat_prediksi.csv`
3. Ada tombol **"Import CSV"** untuk memuat history sebelumnya dari file CSV

Gunakan sel di bawah ini untuk mengembangkan dan menguji logika `session_state` + CSV secara lokal dulu:


In [ ]:
import pandas as pd
import io

# ── Simulasi session_state history (di notebook, pakai dict biasa) ────────────
# Di Streamlit, ganti 'session_history' dengan st.session_state['prediction_history']

session_history = []   # simulasi st.session_state['prediction_history']

def tambah_prediksi(medinc, houseage, prediksi_val):
    """Tambah satu baris prediksi ke history."""
    session_history.append({
        'MedInc': medinc,
        'HouseAge': houseage,
        'Prediksi (×$100K)': round(prediksi_val, 3),
        'Prediksi (USD)': f"${prediksi_val * 100_000:,.0f}"
    })

def export_ke_csv(history):
    """Konversi history ke CSV string (untuk st.download_button)."""
    df = pd.DataFrame(history)
    return df.to_csv(index=False).encode('utf-8')

def import_dari_csv(csv_bytes):
    """Load history dari bytes CSV (untuk st.file_uploader)."""
    df = pd.read_csv(io.BytesIO(csv_bytes))
    return df.to_dict('records')

# ── Test fungsi ────────────────────────────────────────────────────────────────
tambah_prediksi(5.0, 30, 2.345)
tambah_prediksi(8.5, 15, 3.876)
tambah_prediksi(2.1, 45, 1.123)

print("✅ History prediksi (simulasi):")
df_hist = pd.DataFrame(session_history)
print(df_hist.to_string(index=False))

print("\n✅ Export ke CSV (preview 200 karakter pertama):")
csv_bytes = export_ke_csv(session_history)
print(csv_bytes[:200].decode())

print("\n✅ Import dari CSV (re-load):")
history_reloaded = import_dari_csv(csv_bytes)
print(f"Berhasil load {len(history_reloaded)} baris")

# ── TODO: Salin logika ini ke pages/2_Prediksi.py ─────────────────────────────
# 1. Ganti session_history dengan st.session_state['prediction_history']
# 2. Tambahkan st.download_button dengan data=export_ke_csv(st.session_state['prediction_history'])
# 3. Tambahkan st.file_uploader untuk import CSV
print("\n📝 TODO: Implementasikan di pages/2_Prediksi.py!")

---

### Latihan 3 (Tantangan): Full ML App dari Dataset Tim — Deploy ke Streamlit Cloud

Ini adalah **tugas utama** Pertemuan 14. Buat aplikasi ML lengkap menggunakan **dataset proyek tim kamu sendiri** dan deploy ke Streamlit Cloud.

**Persyaratan minimum:**

| Komponen | Deskripsi | Poin |
|----------|-----------|------|
| Multi-page app | Minimal 3 halaman (Home, EDA, Prediksi) | 30 |
| `@st.cache_data` | Digunakan untuk load/preprocess data | 15 |
| `@st.cache_resource` | Digunakan untuk model ML | 15 |
| `st.session_state` | Minimal 1 penggunaan bermakna | 10 |
| Visualisasi | Minimal 2 chart berbeda di halaman EDA | 15 |
| Deploy | App berjalan di Streamlit Cloud | 15 |
| **Total** | | **100** |

**Bonus poin:**
- Tambahkan halaman About/Info Tim (+5)
- Gunakan `st.tabs` atau `st.expander` (+5)
- Tambahkan download button untuk hasil prediksi (+5)

**Cara submit:**
1. Deploy app ke Streamlit Cloud
2. Copy URL app (format: `https://username-appname.streamlit.app`)
3. Submit URL ke LMS beserta link GitHub repo

**Deadline:** Lihat di LMS/info dosen


In [ ]:
# LATIHAN 3 (TANTANGAN): Starter code untuk app ML tim kamu
# Jalankan sel ini untuk memahami struktur yang diperlukan, 
# lalu buat versi sendiri dengan dataset tim.

print("=" * 60)
print("CHECKLIST LATIHAN 3 — Full ML App Tim")
print("=" * 60)

checklist = [
    ("Home.py", "Welcome page dengan dataset overview"),
    ("pages/1_EDA.py", "Eksplorasi data dengan minimal 2 visualisasi"),
    ("pages/2_Prediksi.py", "Input fitur → prediksi → tampilkan hasil"),
    ("pages/3_About.py", "Info tim dan deskripsi project"),
    ("requirements.txt", "Semua library yang dipakai"),
    (".streamlit/config.toml", "Konfigurasi tema (opsional)"),
    ("GitHub repo", "Kode sudah di-push ke GitHub"),
    ("Streamlit Cloud", "App sudah deploy dan bisa diakses via URL"),
]

for i, (item, desc) in enumerate(checklist, 1):
    print(f"  {i}. [ ] {item}")
    print(f"        → {desc}")

print("\n" + "=" * 60)
print("Setelah semua checklist selesai, submit URL app kamu!")
print("=" * 60)

# Template struktur folder untuk dataset tim
print("\n📁 Template struktur folder yang direkomendasikan:")
print("""
my-ml-app/
├── Home.py
├── pages/
│   ├── 1_EDA.py
│   ├── 2_Prediksi.py
│   └── 3_About.py
├── data/
│   └── dataset_tim.csv          ← dataset tim kamu
├── .streamlit/
│   └── config.toml
└── requirements.txt
""")

---

## 📝 Ringkasan Pertemuan 14

```
ALUR LENGKAP: DATA → MODEL → APP → DEPLOY

[P9-P11] DATA & MODEL                   [P13] STREAMLIT DASAR
─────────────────────                   ─────────────────────
Dataset (CSV / sklearn)                 Widgets: slider, button
    ↓                                   Layout: columns, sidebar
Preprocessing                           Visualisasi: st.pyplot
    ↓                                   Demo EDA interaktif
Model Training
    ↓
Evaluasi (MAE, RMSE, R²)                [P14 — HARI INI]
                                        ─────────────────────
                                        @st.cache_data        → load data cepat
                                        @st.cache_resource    → model di-cache
                                        st.session_state      → history prediksi
                                        Multi-page app        → Home + pages/
                                             ↓
                                        GitHub repository
                                             ↓
                                        DEPLOY → Streamlit Cloud URL 🚀
```

### Konsep Kunci yang Dipelajari

| Konsep | Fungsi | Kapan Dipakai |
|--------|--------|---------------|
| `@st.cache_data` | Cache hasil fungsi (data/DataFrame) | Load CSV, preprocessing |
| `@st.cache_resource` | Cache objek berat (model ML) | `model.fit()`, DB connection |
| `st.session_state` | Variabel persist antar re-run | Counter, history, login state |
| `pages/` folder | Multi-page navigation otomatis | App dengan banyak halaman |
| `requirements.txt` | Daftar dependensi untuk deploy | Wajib ada sebelum deploy |

### Perbandingan Platform Deploy

| Platform | Gratis | Mudah | Kontrol | Cocok untuk |
|----------|--------|-------|---------|-------------|
| **Streamlit Cloud** | ✅ | ⭐⭐⭐ | ⭐ | Prototyping, tugas kuliah |
| **Hugging Face Spaces** | ✅ | ⭐⭐ | ⭐⭐ | ML/AI apps, showcase |
| **Railway** | Trial | ⭐⭐ | ⭐⭐⭐ | App web umum |
| **Google Cloud Run** | Trial | ⭐ | ⭐⭐⭐⭐ | Production enterprise |

### Apa Selanjutnya?

Setelah menguasai Streamlit deploy, kamu bisa:
- Tambahkan **autentikasi** (`streamlit-authenticator`)
- Integrasikan **database** (PostgreSQL, SQLite, Firebase)
- Gunakan **Streamlit Components** untuk visualisasi kustom (Plotly, Altair, Folium)
- Buat **REST API** dari model dengan FastAPI — lalu call dari Streamlit
- Deploy dengan **Docker** untuk kontrol infrastruktur penuh

---

**Selamat! Kamu sudah menyelesaikan seluruh rangkaian Python ML (P9–P14).** 🎉  
Dari data mentah → preprocessing → model → evaluasi → aplikasi web → deploy ke cloud.  
Ini adalah skill yang sangat dicari di industri!

---
*Pertemuan 14 — Python Machine Learning | Streamlit Deploy + Full ML App*